# Quantum-Enhanced Agrivoltaics — Colab Simulation Launcher

**Manuscript:** Quantum-Coherent Spectral Engineering (jz-2026-00994t)  
**Solver:** PT-HOPS/SBD via MesoHOPS  

### Usage
- **Google Colab**: upload to [colab.research.google.com](https://colab.research.google.com) and run all cells.
- **Local (MesoHOP-sim env)**: `mamba run -n MesoHOP-sim jupyter notebook`

> ⚠️ **OOM note**: 100 trajectories at L=10 requires ~10 GB RAM per trajectory peak. On Colab (~12 GB), use `N_TRAJ = 5` for a quick test.

## Cell 1 — Mount Drive, clone repo, configure paths

In [ ]:
import importlib.util, subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "Quantum_Agrivoltaic_PT-HOPS"
PACKAGE_PATH = "/content/drive/MyDrive/Colab Notebooks"

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive", force_remount = True)
    os.makedirs(PACKAGE_PATH, exist_ok=True)

    # Prepend persistent package dir so Drive-cached packages are found first
    if PACKAGE_PATH not in sys.path:
        sys.path.insert(0, PACKAGE_PATH)

    # Clone repo if not already present
    repo_root = f"/content/{REPO_NAME}"
    if not os.path.isdir(repo_root):
        token = userdata.get("GITHUB_TOKEN")
        subprocess.run(
            ["git", "clone", "--depth=1",
             f"https://{token}@github.com/NanaEngo/{REPO_NAME}.git", repo_root],
            check=True
        )

    # Set working directory to the simulation framework
    fw_path = os.path.join(repo_root, "Redac_Paper1", "quantum_simulations_framework")
    os.chdir(fw_path)
    if fw_path not in sys.path:
        sys.path.insert(0, fw_path)

    print(f"✅ Drive mounted | CWD: {os.getcwd()}")
    print(f"✅ Persistent packages: {PACKAGE_PATH}")
else:
    print("‹‹ Local environment — no Drive setup needed.")

## Cell 2 — Install dependencies (only if missing)

In [ ]:
# Map: import_name -> pip spec
PACKAGES = {
    "numpy":        "numpy>=2.0,<2.4",
    "scipy":        "scipy>=1.10",
    "pandas":       "pandas>=2.0",
    "matplotlib":   "matplotlib>=3.7",
    "yaml":         "pyyaml>=6.0",
    "numba":        "numba>=0.59",
    "h5py":         "h5py>=3.7",
    "tqdm":         "tqdm>=4.65",
    "psutil":       "psutil>=5.9",
    "scienceplots": "scienceplots>=2.0",
    "qutip":        "qutip>=5.2",
}

def _is_installed(import_name: str) -> bool:
    """True if the package can be found in sys.path (including PACKAGE_PATH)."""
    return importlib.util.find_spec(import_name) is not None

missing = {k: v for k, v in PACKAGES.items() if not _is_installed(k)}

if missing:
    print(f"📦 Installing {len(missing)} missing package(s) to Drive...")
    target = PACKAGE_PATH if "google.colab" in sys.modules else None
    for name, spec in missing.items():
        cmd = [sys.executable, "-m", "pip", "install", spec, "--quiet"]
        if target:
            cmd += ["--target", target]
        subprocess.run(cmd, check=True)
        print(f"  ✓ {spec}")
    print("✅ Installation complete — packages saved to Drive for future sessions.")
else:
    print("✅ All dependencies already present — skipping installation.")

## Cell 3 — Install MesoHOPS (only if missing)

In [ ]:
import shutil, urllib.request, tarfile

if importlib.util.find_spec("mesohops") is not None:
    import mesohops
    print(f"✅ MesoHOPS {mesohops.__version__} already installed — skipping.")
else:
    print("📦 MesoHOPS not found — installing from source...")
    target_tar = "/tmp/mesohops.tar.gz"
    extract_dir = "/tmp/mesohops_src"

    for branch in ("main", "master"):
        url = f"https://github.com/MesoscienceLab/mesohops/archive/refs/heads/{branch}.tar.gz"
        try:
            urllib.request.urlretrieve(url, target_tar)
            print(f"  ✓ Downloaded from branch '{branch}'")
            break
        except Exception as e:
            print(f"  ✗ {branch}: {e}")
    else:
        raise RuntimeError("Could not download MesoHOPS — check network or branch name.")

    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)
    with tarfile.open(target_tar, "r:gz") as tar:
        tar.extractall(extract_dir)

    src = next(os.path.join(extract_dir, d) for d in os.listdir(extract_dir)
               if os.path.isdir(os.path.join(extract_dir, d)))
    target_args = ["--target", PACKAGE_PATH] if "google.colab" in sys.modules else []
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--quiet", src] + target_args, check=True)
    print("✅ MesoHOPS installed successfully.")

## Cell 4 — Import packages & integrity check

In [ ]:
import os
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import numba
import h5py
from tqdm.auto import tqdm
import psutil
import scienceplots
import qutip
import mesohops

print(f"✅ numpy {np.__version__} | qutip {qutip.__version__} | mesohops {mesohops.__version__}")
print(f"✅ CWD: {os.getcwd()}")

## Cell 5 — Load & validate parameters.yaml

In [ ]:
import yaml

with open("parameters.yaml") as f:
    config = yaml.safe_load(f)

print(f"Project : {config['metadata']['project']}")
print(f"L       : {config['dynamics']['hierarchy_depth']}")
print(f"K       : {config['dynamics']['matsubara_truncation']}")
print(f"Δt      : {config['dynamics']['time_step']} fs")
print(f"T       : {config['bath']['temperature']} K")
print(f"n_traj  : {config.get('simulation', {}).get('n_traj', 100)}")

## Cell 6 — Override n_traj for Colab (reduce to avoid OOM)

In [ ]:
# Change to 100 for publication-grade results (requires ~10 GB RAM/traj)
N_TRAJ = 5  # safe for Colab (~12 GB RAM)
config.setdefault("simulation", {})["n_traj"] = N_TRAJ
print(f"n_traj set to {N_TRAJ} for this run.")

## Cell 7 — Check MesoHOPS availability

In [ ]:
from reproducibility.main import check_environment
assert check_environment(), "❌ MesoHOPS not available — re-run Cell 3"

## Cell 8 — Convergence audit (L=9, 10, 11) — optional

In [ ]:
# Uncomment to run; takes ~3 min on Colab
# from reproducibility.main import run_convergence_audit
# audit_data = run_convergence_audit()
# print(f"MAE(L10→L11): {audit_data['audit_mae_10_11']:.2e}")
print("ℹ️  Convergence audit skipped (uncomment to run).")

## Cell 9 — Run simulation & generate figures

In [ ]:
import os
import glob
import matplotlib.image as mpimg
from reproducibility.main import run_full_fmo_simulation, generate_figures

sim_results, time_points = run_full_fmo_simulation(config)
print("✅ Simulation complete.")

generate_figures(config, sim_results, time_points)

jpcl_dir = os.path.abspath("../../Theory_Journals/JPCL")
for pattern, title in [
    ("Quantum_dynamics_*.png",              "Figure 1 — Quantum Dynamics"),
    ("ETR_Under_Environmental_Effects*.png", "Figure 2 — Environmental Robustness"),
]:
    files = sorted(glob.glob(os.path.join(jpcl_dir, pattern)))
    if files:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(mpimg.imread(files[-1]))
        ax.axis("off")
        ax.set_title(title)
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️  {title} not found in {jpcl_dir}")